# Filtro de calidad — calibración de umbrales

Calibra los umbrales de `src/common/quality_filter.py`:

1. **Medida de nitidez.** En el EDA las fotos "más borrosas" resultaron ser las más oscuras, porque la varianza del
   Laplaciano baja con poco contraste. Ahora se divide entre el brillo y se deja fuera la orilla de la retina.
2. **Fotos degradadas a propósito.** El dataset casi no tiene fotos ilegibles (es de una competencia), así que un
   percentil del dataset solo rechazaría fotos buenas. Los umbrales se eligen con fotos buenas a las que se les agrega
   desenfoque u oscuridad: lo que pase con ellas es lo que pasará con una foto mala que llegue a la API.
3. **Revisar a quién rechaza** dentro del dataset: el filtro no debe castigar a una clase ni a una cámara.
4. **Imprimir las constantes** para copiarlas a `quality_filter.py`.

**Cómo correrlo (SageMaker Studio):** *Run All*. La primera vez mide las 3,662 fotos y 150 fotos degradadas (unos
minutos) y lo guarda en `DATA_DIR`. Para ajustar umbrales solo hay que cambiar la celda de la sección 4 y volver a
correr de ahí para abajo.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass

from src.common import quality_filter as qf
from src.common.preprocess import crop_retina, load_image

DATA_DIR = Path(os.environ.get("DATA_DIR", REPO / "data" / "aptos")).expanduser()
CLASS_NAMES = {0: "Sin retinopatía", 1: "Leve", 2: "Moderada", 3: "Severa", 4: "Proliferativa"}
plt.rcParams["figure.dpi"] = 90

labels = pd.read_csv(DATA_DIR / "train.csv")
old = pd.read_csv(DATA_DIR / "image_stats.csv")[["id_code", "height", "width", "blur", "brightness"]]
old = old.rename(columns={"blur": "old_blur", "brightness": "old_brightness"})
splits = pd.read_csv(REPO / "data" / "splits.csv")[["id_code", "split"]]
# Si cambia cómo se mide en quality_filter.py, se cambia VERSION y se vuelve a medir todo
VERSION = f"size={qf.QUALITY_SIZE},margin={qf.EDGE_MARGIN},nitidez=lap/brillo"
print(f"DATA_DIR = {DATA_DIR} · {len(labels)} imágenes")


def load(image_id):
    return load_image(DATA_DIR / "train_images" / f"{image_id}.png")


def measure_cached(path, ids, measure_fn):
    """Mide en paralelo las fotos de `ids` que falten en `path` y las guarda ahí. measure_fn regresa una lista de filas."""
    done = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["id_code", "version"])
    done = done[done.version == VERSION]
    missing = sorted(set(ids) - set(done.id_code))
    if missing:
        print(f"Midiendo {len(missing)} fotos...")
        with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
            rows = [row for rows in pool.map(measure_fn, missing) for row in rows]
        done = pd.concat([done, pd.DataFrame(rows).assign(version=VERSION)], ignore_index=True)
        done.to_csv(path, index=False)
    return done[done.id_code.isin(set(ids))].drop(columns="version")

## 1. Medir todas las fotos

In [ ]:
scores = measure_cached(DATA_DIR / "quality_scores.csv", labels.id_code,
                        lambda i: [{"id_code": i, **qf.quality_scores(load(i))}])

df = labels.merge(scores, on="id_code").merge(old, on="id_code").merge(splits, on="id_code", how="left")
df["split"] = df.split.fillna("quitada (duplicado)")
df["resolution"] = df.height.astype(int).astype(str) + "×" + df.width.astype(int).astype(str)
df[["brightness", "sharpness", "old_blur"]].describe(percentiles=[0.005, 0.01, 0.05, 0.5]).round(2)

## 2. Medida anterior contra la nueva

Si la nueva medida funciona, la nitidez ya no depende del brillo: en la gráfica de la derecha la nube debe quedar
plana (correlación cerca de 0), sin que las fotos oscuras queden todas abajo ni todas arriba.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col, title in [(axes[0], "old_blur", "Antes: varianza del Laplaciano"),
                       (axes[1], "sharpness", "Ahora: varianza del Laplaciano / brillo")]:
    ax.scatter(df.brightness, df[col], s=4, alpha=0.4, color="#2e75b6")
    ax.set_yscale("log"); ax.set_xlabel("brillo de la retina"); ax.set_ylabel("nitidez (escala log)")
    rho = df.brightness.corr(df[col], method="spearman")
    ax.set_title(f"{title}\ncorrelación con el brillo: {rho:.2f}")
plt.tight_layout()
plt.show()

In [ ]:
LABELS = {"brightness": "brillo", "sharpness": "nitidez", "old_blur": "Lapl."}
FORMATS = {"brightness": ".0f", "sharpness": ".2f", "old_blur": ".1f"}


def show_row(rows, title, value_cols=("brightness", "sharpness"), n=6):
    rows = rows.head(n)
    if rows.empty:
        print(f"{title}: ninguna")
        return
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3.1))
    for ax in np.atleast_1d(axes):
        ax.axis("off")
    for ax, (_, r) in zip(np.atleast_1d(axes), rows.iterrows()):
        ax.imshow(crop_retina(load(r.id_code)))
        values = " · ".join(f"{LABELS[c]} {r[c]:{FORMATS[c]}}" for c in value_cols)
        ax.set_title(f"clase {r.diagnosis}\n{values}", fontsize=8)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


show_row(df.nsmallest(6, "old_blur"), "Las 6 'más borrosas' con la medida ANTERIOR", ("brightness", "old_blur"))
show_row(df.nsmallest(6, "sharpness"), "Las 6 más borrosas con la medida NUEVA")
show_row(df.nsmallest(6, "brightness"), "Las 6 más oscuras")

## 3. Fotos degradadas a propósito

150 fotos del split de prueba, cada una en 5 versiones. El desenfoque se mide en pixeles de la retina llevada a
512 × 512, así es igual de fuerte en una cámara de 1,000 px que en una de 4,000 px:

- **desenfoque σ 1:** se ve más suave, pero los exudados y microaneurismas todavía se distinguen. Es el límite.
- **desenfoque σ 2:** las lesiones chicas desaparecen. **Debe rechazarse.**
- **oscura × 0.5:** oscura pero legible. No debe salir "borrosa".
- **oscura × 0.25:** casi negra. **Debe rechazarse** por oscura.

In [ ]:
KINDS = ["original", "desenfoque σ 1", "desenfoque σ 2", "oscura × 0.5", "oscura × 0.25"]


def degrade(img, kind):
    if kind == "original":
        return img
    value = float(kind.split()[-1])
    if kind.startswith("desenfoque"):
        diameter = max(crop_retina(img).shape[:2])
        return cv2.GaussianBlur(img, (0, 0), value * diameter / qf.QUALITY_SIZE)
    return np.clip(img.astype(np.float32) * value, 0, 255).astype(np.uint8)


def measure_degraded(image_id):
    img = load(image_id)
    return [{"id_code": image_id, "kind": k, **qf.quality_scores(degrade(img, k))} for k in KINDS]


test = df[df.split == "test"]
test_ids = test.sample(min(150, len(test)), random_state=0).id_code
degraded = measure_cached(DATA_DIR / "quality_degraded.csv", test_ids, measure_degraded)
degraded.groupby("kind")[["brightness", "sharpness"]].describe(percentiles=[0.05, 0.5, 0.95]).round(2).loc[KINDS]

## 4. Elegir umbrales

**Ajustar aquí.** La idea: que "desenfoque σ 2" y "oscura × 0.25" se rechacen casi siempre, y el dataset (fotos
reales que sí se leen) casi nunca. En las gráficas, la línea roja debe quedar entre la curva del dataset y la de las
fotos que hay que rechazar.

In [ ]:
BRIGHTNESS_THRESHOLD = 25.0
SHARPNESS_THRESHOLD = 0.5


def reason(frame):
    return np.select([frame.brightness < BRIGHTNESS_THRESHOLD, frame.sharpness < SHARPNESS_THRESHOLD],
                     ["oscura", "borrosa"], default="ok")


df["reason"] = reason(df)
degraded["reason"] = reason(degraded)

DATASET = f"dataset ({len(df):,})"
COLORS = {DATASET: "#555555", "desenfoque σ 1": "#f2a541", "desenfoque σ 2": "#c62828",
          "oscura × 0.5": "#7fb3e0", "oscura × 0.25": "#1f4e79"}
groups = {DATASET: df, **{k: degraded[degraded.kind == k] for k in KINDS[1:]}}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, names, thr, bins in [
        (axes[0], "sharpness", [DATASET, "desenfoque σ 1", "desenfoque σ 2", "oscura × 0.5"],
         SHARPNESS_THRESHOLD, np.logspace(-1.3, 1.6, 60)),
        (axes[1], "brightness", [DATASET, "oscura × 0.5", "oscura × 0.25"],
         BRIGHTNESS_THRESHOLD, np.linspace(0, 160, 65))]:
    for name in names:
        ax.hist(groups[name][col].clip(bins[0], bins[-1]), bins=bins, density=True, histtype="step", lw=2,
                color=COLORS[name], label=name)
    ax.axvline(thr, color="red", ls="--", lw=1.5, label=f"umbral {thr}")
    ax.set_xlabel(LABELS[col]); ax.set_yticks([]); ax.legend(fontsize=8)
axes[0].set_xscale("log")
axes[0].set_title("Nitidez"); axes[1].set_title("Brillo")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({name: {"% rechazada": 100 * (g.reason != "ok").mean(),
                               "% oscura": 100 * (g.reason == "oscura").mean(),
                               "% borrosa": 100 * (g.reason == "borrosa").mean()} for name, g in groups.items()}).T
summary.round(1)

### ¿A quién rechaza dentro del dataset?

Las fotos del dataset que se rechacen deberían ser de verdad malas. Si el filtro rechaza mucho más una clase o una
cámara, el sistema pediría repetir la foto más seguido a ciertos pacientes o clínicas.

In [ ]:
rejected = df.reason != "ok"
print(f"Se rechaza el {100 * rejected.mean():.2f}% del dataset ({rejected.sum()} fotos):", df.reason[rejected].value_counts().to_dict())
show_row(df[df.reason == "oscura"].sort_values("brightness"), "Fotos del dataset rechazadas por OSCURAS")
show_row(df[df.reason == "borrosa"].sort_values("sharpness"), "Fotos del dataset rechazadas por BORROSAS")
if rejected.any():
    print("Rechazadas por clase:", df[rejected].diagnosis.value_counts().sort_index().to_dict(),
          "· total por clase:", df.diagnosis.value_counts().sort_index().to_dict())
    print("Rechazadas por resolución:", df[rejected].resolution.value_counts().to_dict())

## 5. Así se ven las fotos degradadas

Tres fotos de prueba con el veredicto del filtro (verde = pasa, rojo = se rechaza).

In [ ]:
sample = degraded.id_code.drop_duplicates().sample(3, random_state=1)
fig, axes = plt.subplots(len(sample), len(KINDS), figsize=(2.7 * len(KINDS), 3 * len(sample)), squeeze=False)
for row, image_id in enumerate(sample):
    img = load(image_id)
    for col, kind in enumerate(KINDS):
        out = degrade(img, kind)
        res = qf.check_quality(out, BRIGHTNESS_THRESHOLD, SHARPNESS_THRESHOLD)
        ax = axes[row, col]
        ax.imshow(crop_retina(out)); ax.set_xticks([]); ax.set_yticks([])
        color = "#2e7d32" if res.ok else "#c62828"
        for side in ax.spines.values():
            side.set_edgecolor(color); side.set_linewidth(4)
        verdict = "pasa" if res.ok else f"rechaza: {res.reason}"
        ax.set_title(f"{kind}\n{verdict}\nbrillo {res.scores['brightness']:.0f} · nitidez {res.scores['sharpness']:.2f}", fontsize=8, color=color)
plt.tight_layout()
plt.show()

## 6. Constantes para `quality_filter.py`

In [ ]:
print("# Copiar a src/common/quality_filter.py (salen de notebooks/quality.ipynb)")
print(f"BRIGHTNESS_THRESHOLD = {BRIGHTNESS_THRESHOLD}   # menor = oscura")
print(f"SHARPNESS_THRESHOLD = {SHARPNESS_THRESHOLD}     # menor = borrosa")

## 7. Conclusiones

Llenar después de correr el notebook:

- **Medida nueva contra la anterior:** …
- **Umbrales elegidos:** …
- **A quién rechaza:** …
- **Prueba con fotos degradadas:** …